# 03 — GNU Go development and frozen final evaluation

This notebook measures the native Queen/Worker/Scout AHR against the pinned GNU Go 3.8 **level 1** executable. Run All performs short recovery and protocol checks only. Long evaluations are explicitly invoked by notebook 04.

**Inputs:** native AHR training checkpoint; original tagged Go rules/verifier and GNU replay definitions. **Outputs:** split-specific JSON results, protected model copy, resumable evaluation checkpoints and latency/failure analysis. **Runtime:** Run All normally under one minute; 32 development games approximately 1 hour and final 200 approximately 4–8 hours, depending on game length. **Rules:** 9×9, Chinese area scoring, positional superko, forbidden suicide, komi 7.5, no handicap, two-ply seeded openings paired across colors. AHR decision deadline is exactly one second after model warmup. GNU has a five-second exchange allowance; actual duration is reported. No equal-compute claim is made.

**Recovery:** checkpoints live in `checkpoints/<eval_run>/`. Restart the project kernel, Run All in 00 and import only `ahr-export` cells from 01–03, then repeat `v4_run_gnu_evaluation(project, training_run, eval_run, split=..., games=..., checkpoint_path=..., device='cuda:1')` with the original arguments. The manifest fixes the model, sources, protocol, complete chronology, every RNG, opening cursor and active game. Invalid manifests are rejected. One writer is enforced. Game/stage boundaries and at most 32 moves or five minutes trigger atomic commits; latest two plus best are retained. The protected `frozen_model.pt` survives training checkpoint pruning.

Development and final reports use separate directories and seed domains. The first final request freezes one candidate and 200 games before any result. Final games are never training inputs. UNKNOWN is preserved for cutoff, deadline or illegal emission; it is never counted as a proved loss. Observed half-wins and statistically supported performance above half are reported separately. No automatic player promotion occurs.

In [ ]:
from pathlib import Path
import nbformat
project = Path.cwd().resolve()
while not (project/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if project.parent == project: raise RuntimeError('Run inside the AHR Go workspace')
    project = project.parent
book = nbformat.read(project/'experiments/ahr_native_v4/00_shared.ipynb', 4)
for cell in book.cells:
    if cell.cell_type == 'code' and 'ahr-export' in cell.metadata.get('tags', []):
        exec(compile(cell.source, '00_shared:ahr-export', 'exec'), globals())
v4_load_upstream(project, globals())
for name in ('01_native_ahr_and_tactics.ipynb', '02_competitive_training.ipynb'):
    path = project/V4_ROOT/name
    if path.exists(): v4_import_exports(path, globals())


## Freeze protocol, opening schedule and uncertainty definitions

In [ ]:
from dataclasses import dataclass, asdict
from collections import Counter
import math, random, re, statistics, time, copy, hashlib, json, uuid
import numpy as np
import torch

V4_EVAL_SCHEMA = 'ahr-native-v4-gnu-evaluation-v1'
V4_EVAL_RESUME = '03: fresh project kernel; import only tagged 00–03 definitions; repeat v4_run_gnu_evaluation with unchanged arguments'
V4_EVAL_SEED_DOMAINS = {'development': 410000000, 'final': 710000000, 'recovery_fixture': 910000000}

@dataclass(frozen=True)
class V4GNUProtocol:
    split: str = 'development'
    games: int = 32
    seed: int = 410000001
    size: int = 9
    komi: float = 7.5
    opening_plies: int = 2
    max_plies: int = 512
    ahr_budget_s: float = 1.0
    gnu_budget_s: float = 5.0
    gnu_command_timeout_s: float = 3.0
    level: int = 1
    alpha: float = .05
    def __post_init__(self):
        if self.split not in V4_EVAL_SEED_DOMAINS: raise ValueError('Unrecognized data split')
        if self.split == 'final' and self.games != 200: raise ValueError('Final benchmark is exactly 200 attempts')
        if self.split == 'development' and self.games not in (32,64): raise ValueError('Development blocks contain 32 or 64 games')
        if type(self.games) is not int or self.games < 2 or self.games % 2: raise ValueError('Require complete black/white opening pairs')
        if self.size != 9 or self.komi != 7.5 or self.level != 1: raise ValueError('Pinned no-handicap 9x9/7.5/level1 protocol')
        if self.ahr_budget_s != 1.0 or self.gnu_budget_s != 5.0 or self.gnu_command_timeout_s != 3.0:
            raise ValueError('Pinned decision and exchange allowances')
        if not 0 <= self.opening_plies < self.max_plies or self.max_plies > 512: raise ValueError('Invalid chronology bounds')
        if not 0 < self.alpha < 1: raise ValueError('Invalid confidence parameter')

def v4_eval_path(project, split, name):
    if split not in V4_EVAL_SEED_DOMAINS or Path(name).name != name: raise ValueError('Unsafe split/report name')
    path = Path(project)/'reports/ahr_native_v4'/split/name
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

def v4_eval_schedule(protocol):
    domain = V4_EVAL_SEED_DOMAINS[protocol.split]
    rng = random.Random(protocol.seed)
    seeds = rng.sample(range(domain,domain+100000000),protocol.games//2)
    openings = []
    for seed in seeds:
        board = initial_state(protocol.size,protocol.komi)
        local = random.Random(seed); actions = []
        for _ in range(protocol.opening_plies):
            legal = [a for a in legal_actions(board) if a != board.pass_action]
            action = local.choice(legal); actions.append(action); board = play(board,action)
        openings.append({'seed':seed,'actions':actions,'state':to_dict(board)})
    jobs = [{'index':2*i+j,'block':i,'seed':s,'candidate_color':color}
            for i,s in enumerate(seeds) for j,color in enumerate((BLACK,WHITE))]
    return openings,jobs

def v4_eval_identity(project, protocol, players, candidate_receipt=None):
    if set(players) != {'candidate','reference'}: raise ValueError('Require native AHR and one reference')
    for player in players.values(): player.check_frozen()
    openings,jobs = v4_eval_schedule(protocol)
    return json.loads(arena_json({'schema':V4_EVAL_SCHEMA,'protocol':asdict(protocol),
        'rules':RULESET,'handicap':0,'sources':v4_source_identity(project,require_all=False),
        'openings':openings,'jobs':jobs,'players':{name:p.identity() for name,p in players.items()},
        'candidate_receipt':candidate_receipt,
        'gnugo_runtime':'fresh pinned process every call; complete chronology and state-derived seed; no hidden continuing engine state',
        'outcome_source':'independent project area scorer; no GNU final_score or suggested-move labels'}))

def v4_eval_statistics(identity, results):
    protocol = identity['protocol']; planned = protocol['games']
    complete = [r for r in results if r['status'] == 'COMPLETE']
    wins = sum(r['candidate_score'] == 1.0 for r in complete)
    losses = sum(r['candidate_score'] == 0.0 for r in complete)
    draws = sum(r['candidate_score'] == .5 for r in complete)
    unknown = len(results)-len(complete); pending = planned-len(results)
    grouped = {}
    for row in results: grouped.setdefault(row['job']['block'],[]).append(row)
    valid_pairs = [rows for rows in grouped.values() if len(rows)==2 and all(r['status']=='COMPLETE' for r in rows)]
    paired_values = [sum(r['candidate_score']==1.0 for r in pair)/2 for pair in valid_pairs]
    # Each independent opening contributes one bounded [0,1] paired win fraction.
    # No independence assumption between the two colors of one opening is made.
    interval = None
    if paired_values:
        mean = sum(paired_values)/len(paired_values)
        radius = math.sqrt(math.log(2/protocol['alpha'])/(2*len(paired_values)))
        interval = [max(0.,mean-radius),min(1.,mean+radius)]
    all_valid = len(complete)==planned and unknown==0 and pending==0
    margins = [r['black_score']*r['job']['candidate_color'] for r in complete]
    audits = [a for r in results for a in r['moves']]
    latency = {}
    for name,budget in [('candidate',protocol['ahr_budget_s']),('reference',protocol['gnu_budget_s'])]:
        times = [a['elapsed_s'] for a in audits if a['player']==name]
        latency[name] = {'calls':len(times),'mean_s':statistics.mean(times) if times else None,
                         'max_s':max(times) if times else None,'over_budget':sum(t>budget for t in times),
                         'allocated_budget_s':budget}
    return {'games_planned':planned,'games_recorded':len(results),'games_complete':len(complete),
        'wins':wins,'losses':losses,'draws':draws,'unknown':unknown,'pending':pending,
        'win_rate_completed':wins/len(complete) if complete else None,'wins_over_planned':wins/planned,
        'possible_win_rate_bounds':[wins/planned,(wins+unknown+pending)/planned],
        'mean_score_margin':statistics.mean(margins) if margins else None,
        'paired_blocks_complete':len(paired_values),'paired_confidence_interval':interval,
        'confidence_method':'two-sided Hoeffding interval over independent paired opening blocks; conditional on one frozen model and declared opening distribution',
        'confidence_level':1-protocol['alpha'],'all_games_valid':all_valid,
        'observed_half_wins':bool(all_valid and wins>=planned/2),
        'statistically_supported_above_half':bool(all_valid and interval and interval[0]>.5),
        'unknown_by_reason':dict(Counter(r['status'] for r in results if r['status']!='COMPLETE')),
        'latency_by_player':latency,'promotion_effect':'NONE','full_game_optimality':'NOT_ESTABLISHED'}


## Resume exact chronology and commit interruption-safe checkpoints

In [ ]:
def v4_eval_replay(identity, job, actions):
    board = initial_state(identity['protocol']['size'],identity['protocol']['komi'])
    opening = identity['openings'][job['block']]
    for action in opening['actions']: board = play(board,action)
    if to_dict(board)!=opening['state']: raise RuntimeError('Opening chronology changed')
    for action in actions: board = play(board,action)
    return board

def v4_eval_strict_load(run_dir):
    run_dir = Path(run_dir); manifest_path = run_dir/'manifest.json'
    if not manifest_path.exists():
        if list(run_dir.glob('step-*.pt')): raise RuntimeError('Missing evaluation manifest; refusing restart')
        return None
    try:
        manifest = v4_read_json(manifest_path); entries=manifest['entries']; states={}
        if not entries or len({e['file'] for e in entries})!=len(entries): raise ValueError('Bad checkpoint inventory')
        for entry in entries:
            if Path(entry['file']).name!=entry['file']: raise ValueError('Unsafe checkpoint path')
            state=_checkpoint_entry_state(run_dir,entry)
            if state is None: raise ValueError('Invalid committed evaluation checkpoint: '+entry['file'])
            states[entry['file']]=state
        if manifest['latest'] not in states or manifest['best'] not in states: raise ValueError('Missing latest/best')
        latest=states[manifest['latest']]
        if latest['global_step']!=max(s['global_step'] for s in states.values()): raise ValueError('Rewound latest cursor')
        return latest
    except (KeyError,ValueError,TypeError,OSError) as error:
        raise RuntimeError('Invalid evaluation manifest; refusing silent recovery') from error

def v4_eval_validate_snapshot(snapshot,identity,verify_history=True):
    if REQUIRED_CHECKPOINT_KEYS-set(snapshot): raise RuntimeError('Incomplete evaluation resume payload')
    if snapshot.get('schema')!=V4_EVAL_SCHEMA or snapshot['config']!=identity or snapshot.get('identity_sha256')!=v4_digest(identity):
        raise RuntimeError('Frozen model, source, split or protocol changed')
    if any(snapshot[k] is not None for k in ('optimizer','scheduler','scaler','replay')) or snapshot['optimizer_steps']!=0:
        raise RuntimeError('Evaluation contains unexpected training state')
    if snapshot['distributed']!={'world_size':1}: raise RuntimeError('Unexpected evaluation distributed state')
    cursor=snapshot['sampler']['next_game'];jobs=identity['jobs']
    if type(cursor) is not int or not 0<=cursor<=len(jobs) or len(snapshot['results'])!=cursor:
        raise RuntimeError('Evaluation sampler/result cursor mismatch')
    if snapshot['complete']!=(cursor==len(jobs)): raise RuntimeError('Invalid completion marker')
    if type(snapshot['global_step']) is not int or snapshot['global_step']<0: raise RuntimeError('Invalid event cursor')
    for name,weights in snapshot['model'].items():
        if name not in identity['players'] or arena_tensor_digest(weights)!=identity['players'][name]['weights_sha256']:
            raise RuntimeError('Frozen model changed')
    if set(snapshot['model'])!=set(identity['players']) or set(snapshot['runtime'])!=set(identity['players']):
        raise RuntimeError('Model/runtime inventory mismatch')
    for i,row in enumerate(snapshot['results'] if verify_history else []):
        if row['job']!=jobs[i]: raise RuntimeError('Paired result ordering changed')
        board=v4_eval_replay(identity,row['job'],row['actions'])
        if to_dict(board)!=row['final_state']: raise RuntimeError('Result chronology differs from saved board')
        if row['status']=='COMPLETE':
            score=area_score(board);outcome=.5 if score==0 else float(score*row['job']['candidate_color']>0)
            if score!=arena_reference_score(board): raise RuntimeError('Independent final area scorers disagree')
            if not board.terminal or row['black_score']!=score or row['candidate_score']!=outcome:
                raise RuntimeError('Terminal result failed independent scoring')
        elif not row['status'].startswith('UNKNOWN_') or row['black_score'] is not None or row['candidate_score'] is not None:
            raise RuntimeError('UNKNOWN must not be scored as a loss')
    active=snapshot['active']
    if active is not None:
        if cursor>=len(jobs) or active['job']!=jobs[cursor]: raise RuntimeError('Active game cursor changed')
        if len(active['moves'])!=len(active['actions']): raise RuntimeError('Active move audit mismatch')
        if verify_history and to_dict(v4_eval_replay(identity,active['job'],active['actions']))!=active['state']:
            raise RuntimeError('Active game chronology differs from saved board')
    return True

def v4_eval_commit(run_dir,snapshot,players,first=False):
    for player in players.values(): player.check_frozen()
    if first: snapshot['model']={name:p.weights() for name,p in players.items()}
    snapshot['runtime']={name:p.runtime() for name,p in players.items()}
    snapshot['rng']=capture_rng()
    snapshot['metrics']={'recorded_games':len(snapshot['results']),
                         'complete_games':sum(r['status']=='COMPLETE' for r in snapshot['results'])}
    v4_eval_validate_snapshot(snapshot,snapshot['config'],verify_history=False)
    return _save_checkpoint_unlocked(run_dir,snapshot,best=first or snapshot['complete'],resume_cell=V4_EVAL_RESUME)

def v4_eval_finish(snapshot,status):
    active=snapshot['active'];board=from_dict(active['state'])
    score=area_score(board) if status=='COMPLETE' else None
    outcome=(.5 if score==0 else float(score*active['job']['candidate_color']>0)) if score is not None else None
    snapshot['results'].append({'job':active['job'],'actions':active['actions'],'moves':active['moves'],
        'final_state':active['state'],'status':status,'black_score':score,'candidate_score':outcome})
    snapshot['sampler']['next_game']+=1;snapshot['active']=None
    snapshot['complete']=snapshot['sampler']['next_game']==len(snapshot['config']['jobs'])

def v4_eval_run(project,eval_run,identity,players,stop_after_events=None):
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,100}',eval_run): raise ValueError('Simple eval run ID required')
    if stop_after_events is not None and (type(stop_after_events) is not int or stop_after_events<1): raise ValueError('Positive event cutoff required')
    run_dir=Path(project)/'checkpoints'/eval_run
    if identity['protocol']['split']!='recovery_fixture':
        validation=v4_read_json(v4_report_path(project,'arena_validation.json'))
        if validation.get('exact_resume') is not True or validation.get('sources')!=identity['sources']:
            raise RuntimeError('Current exact-resume validation is required before long evaluation')
    with checkpoint_lock(run_dir):
        snapshot=v4_eval_strict_load(run_dir)
        if snapshot is None:
            if (run_dir/'identity.json').exists() and v4_read_json(run_dir/'identity.json')!=identity:
                raise RuntimeError('Frozen evaluation request changed')
            v4_atomic_json(run_dir/'identity.json',identity);v4_seed(identity['protocol']['seed'])
            snapshot={'schema':V4_EVAL_SCHEMA,'identity_sha256':v4_digest(identity),'config':identity,
                'model':{},'runtime':{},'initial_runtime':{name:p.runtime() for name,p in players.items()},
                'optimizer':None,'scheduler':None,'scaler':None,'rng':None,'sampler':{'next_game':0},
                'global_step':0,'optimizer_steps':0,'metrics':{},'replay':None,'distributed':{'world_size':1},
                'results':[],'active':None,'complete':False}
            v4_eval_commit(run_dir,snapshot,players,first=True)
        else:
            v4_eval_validate_snapshot(snapshot,identity)
            if not (run_dir/'identity.json').is_file() or v4_read_json(run_dir/'identity.json')!=identity:
                raise RuntimeError('Committed evaluation identity missing/changed')
            for name,player in players.items(): player.restore(snapshot['model'][name],snapshot['runtime'][name])
            restore_rng(snapshot['rng'])
        events=0;last_commit=time.monotonic();moves_since_commit=0
        while not snapshot['complete']:
            boundary=False
            if snapshot['active'] is None:
                job=identity['jobs'][snapshot['sampler']['next_game']];v4_seed(job['seed'])
                for name,player in players.items():
                    if player.set_runtime_state: player.set_runtime_state(copy.deepcopy(snapshot['initial_runtime'][name]))
                snapshot['active']={'job':job,'state':copy.deepcopy(identity['openings'][job['block']]['state']),
                                    'actions':[],'moves':[]}
                boundary=True
            else:
                active=snapshot['active'];board=from_dict(active['state']);protocol=identity['protocol']
                if board.terminal: v4_eval_finish(snapshot,'COMPLETE');boundary=True
                elif board.move_number>=protocol['max_plies']: v4_eval_finish(snapshot,'UNKNOWN_MOVE_CAP');boundary=True
                else:
                    name='candidate' if board.to_play==active['job']['candidate_color'] else 'reference'
                    player=players[name];budget=protocol['ahr_budget_s'] if name=='candidate' else protocol['gnu_budget_s']
                    device=next(player.model.parameters()).device if player.model is not None else None
                    if device is not None and device.type=='cuda': torch.cuda.synchronize(device)
                    started=time.perf_counter();prepared_at=started;failure=None;successor=None
                    try:
                        if player.prepare_position:
                            player.prepare_position(board,copy.deepcopy(identity['openings'][active['job']['block']]['actions']),copy.deepcopy(active['actions']))
                        prepared_at=time.perf_counter()
                        with torch.inference_mode(): result=player.decide(board,budget)
                        if device is not None and device.type=='cuda': torch.cuda.synchronize(device)
                    except Exception as error:
                        result={'action':None,'exception':type(error).__name__+': '+str(error)[:250]};failure='UNKNOWN_PLAYER_ERROR'
                    elapsed=time.perf_counter()-started
                    if not isinstance(result,dict): result={'action':None};failure='UNKNOWN_PLAYER_ERROR'
                    action=result.get('action')
                    if elapsed>budget or result.get('deadline_missed',False): failure='UNKNOWN_DEADLINE'
                    try:
                        successor=play(board,action)
                        if not verify_go_emission(board,action,successor): raise RuntimeError('Independent verifier rejected transition')
                        legal=True
                    except (IllegalMove,TypeError,ValueError):
                        legal=False;successor=None
                        if failure is None: failure='UNKNOWN_ILLEGAL_MOVE'
                    audit={'player':name,'action':action if type(action) is int else None,'legal':legal,'elapsed_s':elapsed,
                        'allocated_budget_s':budget,'preparation_s':prepared_at-started,
                        'decision_and_exchange_s':max(0.,elapsed-(prepared_at-started)),
                        'deadline_missed':elapsed>budget or bool(result.get('deadline_missed',False)),
                        'failure':failure,'strategic_status':'UNKNOWN',
                        'counters':{k:result[k] for k in ('completed_depth','nodes','model_calls','root_value','reference_seed')
                                    if k in result and type(result[k]) in (int,float,str,bool)},
                        'rng_probe':result.get('rng_probe'),'exception':result.get('exception')}
                    active['moves'].append(audit)
                    if failure: v4_eval_finish(snapshot,failure);boundary=True
                    else:
                        active['actions'].append(action);active['state']=to_dict(successor)
                    moves_since_commit+=1
            snapshot['global_step']+=1;events+=1
            stopping=stop_after_events is not None and events>=stop_after_events
            if boundary or stopping or moves_since_commit>=32 or time.monotonic()-last_commit>=300:
                v4_eval_commit(run_dir,snapshot,players);last_commit=time.monotonic();moves_since_commit=0
                progress=v4_eval_statistics(identity,snapshot['results'])
                v4_atomic_json(run_dir/'progress.json',{'status':'COMPLETE' if snapshot['complete'] else 'IN_PROGRESS',
                    'global_step':snapshot['global_step'],'active_move':len(snapshot['active']['actions']) if snapshot['active'] else None,
                    **progress})
            if stopping: break
        for player in players.values(): player.check_frozen()
        return snapshot,v4_eval_statistics(identity,snapshot['results'])


## Diagnose captured groups and evaluate one protected checkpoint

In [ ]:
def v4_eval_failures(identity,results):
    rows=[]
    for result in results:
        color=result['job']['candidate_color'];board=v4_eval_replay(identity,result['job'],[])
        first_loss=None;total_lost=0;first_atari_ignored=None;depths=[]
        for i,action in enumerate(result['actions']):
            if board.to_play==color:
                endangered=[sorted(g) for g,l in groups_and_liberties(board.board,board.size)
                            if board.board[next(iter(g))]==color and len(l)==1]
                child=play(board,action)
                if endangered and first_atari_ignored is None:
                    unresolved=[g for g in endangered if any(child.board[p]==color for p in g)
                                and len(group_and_liberties(child.board,child.size,next(p for p in g if child.board[p]==color))[1])==1]
                    if unresolved: first_atari_ignored={'move_number':board.move_number,'action':action,'groups':unresolved,
                        'interpretation':'Group remained in atari after this move; this is a diagnostic flag, not proof that defense was best'}
            else: child=play(board,action)
            removed=[p for p in range(board.size**2) if board.board[p]==color and child.board[p]!=color]
            if removed:
                total_lost+=len(removed)
                if first_loss is None:
                    seen=set();groups=[]
                    for p in removed:
                        if p not in seen:
                            group,liberties=group_and_liberties(board.board,board.size,p);seen.update(group)
                            groups.append({'stones':sorted(group),'liberties_before_capture':sorted(liberties)})
                    first_loss={'move_number':board.move_number,'capturing_action':action,'groups':groups,'stones_lost':len(removed)}
            board=child
        for audit in result['moves']:
            if audit['player']=='candidate' and 'completed_depth' in audit['counters']:
                depths.append(audit['counters']['completed_depth'])
        rows.append({'game_index':result['job']['index'],'candidate_color':color,'status':result['status'],
            'won':result['candidate_score']==1 if result['status']=='COMPLETE' else None,
            'first_lost_group':first_loss,'stones_captured_over_game':total_lost,
            'surviving_candidate_stones':sum(v==color for v in board.board),
            'first_unresolved_atari':first_atari_ignored,'completed_depth_counts':dict(Counter(depths))})
    return {'games':rows,'games_with_zero_surviving_stones':sum(r['surviving_candidate_stones']==0 for r in rows),
        'scope':'Descriptive replay diagnostics; early capture and unresolved atari do not establish the causal losing move'}

def v4_eval_pin_model(project,training_run,eval_run,checkpoint_path,device):
    run_dir=Path(project)/'checkpoints'/eval_run;pin=run_dir/'frozen_model.pt';receipt_path=run_dir/'frozen_model.json'
    with checkpoint_lock(Path(project)/'checkpoints'/('pin-'+eval_run)):
        if pin.exists()!=receipt_path.exists(): raise RuntimeError('Incomplete frozen model pin; inspect rather than restart')
        if pin.exists():
            declared=v4_read_json(receipt_path)
            if declared['training_run']!=training_run or file_sha256(pin)!=declared['checkpoint_sha256']:
                raise RuntimeError('Protected model pin identity changed')
            if checkpoint_path is not None:
                supplied=Path(checkpoint_path)
                if not supplied.is_absolute(): supplied=Path(project)/supplied
                supplied=supplied.resolve()
                if not supplied.is_relative_to(Path(project).resolve()): raise RuntimeError('Pinned checkpoint must belong to this workspace')
                allowed_missing=(Path(project)/declared['original_checkpoint_path']).resolve()
                if not supplied.exists() and supplied!=allowed_missing: raise RuntimeError('Requested missing checkpoint is not the original protected source')
                if supplied.exists() and file_sha256(supplied)!=declared['checkpoint_sha256']:
                    raise RuntimeError('Requested checkpoint differs from frozen model')
            model,receipt=v4_load_model(project,training_run,device=device,checkpoint_path=str(pin.relative_to(project)))
        else:
            model,receipt=v4_load_model(project,training_run,device=device,checkpoint_path=checkpoint_path)
            source=Path(receipt['checkpoint_path'])
            if not source.is_absolute(): source=Path(project)/source
            run_dir.mkdir(parents=True,exist_ok=True)
            atomic_write_bytes(pin,source.read_bytes())
            if file_sha256(pin)!=receipt['checkpoint_sha256']: raise RuntimeError('Protected model copy hash mismatch')
            v4_atomic_json(receipt_path,{'training_run':training_run,'checkpoint_sha256':receipt['checkpoint_sha256'],
                'original_checkpoint_path':receipt['checkpoint_path'],'protected_checkpoint_path':str(pin.relative_to(project))})
        if receipt.get('purpose') not in ('go_training','native_go_training') or receipt.get('optimizer_steps',0)<1:
            raise RuntimeError('Strength evaluation requires actual AHR training, not synthetic recovery fixtures')
        frozen={k:receipt.get(k) for k in ('checkpoint_sha256','global_step','optimizer_steps','games_finished','config','model_config','source_identity','purpose')}
        frozen.update(training_run=training_run,checkpoint_path=str(pin.relative_to(project)))
        return model,frozen

def v4_run_gnu_evaluation(project,training_run,eval_run,split='development',games=32,checkpoint_path=None,
                         device='cuda:1',stop_after_events=None):
    project=Path(project).resolve()
    if split not in ('development','final'): raise ValueError('Public arena accepts development or final only')
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,100}',eval_run): raise ValueError('Simple evaluation run ID required')
    model,receipt=v4_eval_pin_model(project,training_run,eval_run,checkpoint_path,device)
    model.eval();model.requires_grad_(False)
    search=V4SearchConfig(time_budget_s=1.0)
    # Warm model and search before any measured call, including after a resumed VM/kernel.
    with torch.inference_mode():
        warm_state=initial_state(9,7.5)
        for _ in range(2): v4_choose_move(model,warm_state,search)
    if next(model.parameters()).device.type=='cuda': torch.cuda.synchronize(next(model.parameters()).device)
    candidate=FrozenArenaPlayer('candidate',lambda state,budget:v4_choose_move(model,state,search),
        {'search':asdict(search),'ahr_budget_s':1.0,'architecture':'native Queen/Worker/Scout','jepa':False},
        {'kind':'ahr_native_v4_checkpoint',**receipt},model=model)
    domain=V4_EVAL_SEED_DOMAINS[split]
    # Opening seeds depend on eval_run and split, never on observed results or training RNG.
    seed=domain+int(hashlib.sha256((split+':'+eval_run).encode()).hexdigest()[:12],16)%100000000
    protocol=V4GNUProtocol(split=split,games=games,seed=seed)
    reference=arena_gnugo_player(project,'reference',level=1,seed=seed,command_timeout_s=3.0)
    reference.config['allocated_exchange_budget_s']=5.0
    players={'candidate':candidate,'reference':reference}
    identity=v4_eval_identity(project,protocol,players,receipt)
    request_path=v4_eval_path(project,split,eval_run+'-request.json')
    with checkpoint_lock(project/'checkpoints'/('request-'+eval_run)):
        if request_path.exists() and v4_read_json(request_path)!=identity: raise RuntimeError('Frozen request changed')
        if not request_path.exists(): v4_atomic_json(request_path,identity)
    if split=='final':
        lockbox_path=project/'config/ahr_native_v4_final_lockbox.json'
        with checkpoint_lock(project/'checkpoints'/'ahr-native-v4-final-lockbox'):
            lockbox={'eval_run':eval_run,'identity_sha256':v4_digest(identity),'identity':identity,
                'access':'final-only; excluded from trainer and development feedback','promotion_effect':'NONE'}
            if lockbox_path.exists() and v4_read_json(lockbox_path)!=lockbox:
                raise RuntimeError('Final candidate/protocol already frozen; final results cannot select a new candidate')
            if not lockbox_path.exists(): v4_atomic_json(lockbox_path,lockbox)
    snapshot,stats=v4_eval_run(project,eval_run,identity,players,stop_after_events=stop_after_events)
    report_path=v4_eval_path(project,split,eval_run+'-results.json')
    report={'schema':V4_EVAL_SCHEMA,'status':'COMPLETE' if snapshot['complete'] else 'IN_PROGRESS',
        'split':split,'eval_run':eval_run,'training_run':training_run,'candidate':receipt,'identity_sha256':v4_digest(identity),
        'statistics':stats,'failure_analysis':v4_eval_failures(identity,snapshot['results']),
        'results':snapshot['results'],'final_data_used_for_training':False,
        'selected_player_changed':False,'unknown_semantics':'UNKNOWN is unscored; denominator and possible bounds include all planned attempts'}
    v4_atomic_json(report_path,report)
    returned={k:receipt[k] for k in ('checkpoint_path','checkpoint_sha256','optimizer_steps','games_finished')}
    returned.update(schema=V4_EVAL_SCHEMA,status=report['status'],split=split,eval_run=eval_run,training_run=training_run,
                    report_path=str(report_path.relative_to(project)),**stats)
    v4_atomic_json(v4_eval_path(project,split,eval_run+'-receipt.json'),returned)
    return returned


## Short recovery fixtures and actual GNU protocol checks

In [ ]:
def v4_eval_fixture_players(mode='normal'):
    def decide(state,budget):
        probe=[random.random(),float(np.random.random()),float(torch.rand(()))]
        if mode=='illegal': action=-7
        elif mode=='cutoff': action=next(a for a in legal_actions(state) if a!=state.pass_action)
        elif mode=='deadline': return {'action':state.pass_action,'deadline_missed':True,'rng_probe':probe}
        elif state.move_number>=4: action=state.pass_action
        else:
            choices=[a for a in legal_actions(state) if a!=state.pass_action]
            action=choices[int(probe[0]*len(choices))%len(choices)]
        return {'action':action,'rng_probe':probe,'completed_depth':0,'nodes':0,'deadline_missed':False}
    return {name:FrozenArenaPlayer(name,decide,{'fixture':mode},{'kind':'fixture','purpose':'recovery_only'})
            for name in ('candidate','reference')}

def v4_eval_semantic(snapshot):
    rows=[]
    for row in snapshot['results']:
        clean=copy.deepcopy(row)
        for audit in clean['moves']:
            for key in ('elapsed_s','preparation_s','decision_and_exchange_s'): audit.pop(key,None)
        rows.append(clean)
    return {'results':rows,'sampler':snapshot['sampler'],'global_step':snapshot['global_step'],'complete':snapshot['complete']}

def v4_eval_fixture_subprocess(project,eval_run,identity,stop_after_events=None):
    import subprocess
    project=Path(project).resolve()
    request=project/'reports/ahr_native_v4/recovery_fixture'/(eval_run+'-fixture-request.json')
    v4_atomic_json(request,identity)
    program = """import json,sys
from pathlib import Path
import nbformat
project=Path(sys.argv[1]);request=Path(sys.argv[2]);run=sys.argv[3]
namespace={'__name__':'__main__'}
book=nbformat.read(project/'experiments/ahr_native_v4/00_shared.ipynb',4)
for cell in book.cells:
 if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'00:export','exec'),namespace)
namespace['v4_load_upstream'](project,namespace)
namespace['v4_import_exports'](project/'experiments/ahr_native_v4/03_gnugo_evaluation.ipynb',namespace)
identity=json.loads(request.read_text());cutoff=None if sys.argv[4]=='None' else int(sys.argv[4])
snapshot,stats=namespace['v4_eval_run'](project,run,identity,namespace['v4_eval_fixture_players'](),stop_after_events=cutoff)
print('V4_ARENA_FIXTURE='+namespace['v4_digest'](namespace['v4_eval_semantic'](snapshot)))
"""
    process=subprocess.run([str(project/'.venv/bin/python'),'-c',program,str(project),str(request),eval_run,str(stop_after_events)],
                           cwd=project,capture_output=True,text=True,timeout=90,check=True)
    markers=[line.split('=',1)[1] for line in process.stdout.splitlines() if line.startswith('V4_ARENA_FIXTURE=')]
    if len(markers)!=1:raise RuntimeError('Fresh arena recovery worker did not emit exact-state digest')
    return markers[0]

def v4_validate_arena_resume(project,device='cuda:1'):
    project=Path(project).resolve();suffix=uuid.uuid4().hex[:10]
    protocol=V4GNUProtocol(split='recovery_fixture',games=2,seed=910000017,opening_plies=0,max_plies=16)
    players=v4_eval_fixture_players();identity=v4_eval_identity(project,protocol,players)
    whole,_=v4_eval_run(project,'ahr-v4-arena-whole-'+suffix,identity,players)
    split_run='ahr-v4-arena-resume-'+suffix
    partial,_=v4_eval_run(project,split_run,identity,v4_eval_fixture_players(),stop_after_events=3)
    if partial['active'] is None or len(partial['active']['actions'])!=2: raise AssertionError('Did not interrupt midgame')
    # Deliberately perturb all RNGs before restoration.
    random.random();np.random.random();torch.rand(())
    fresh_digest=v4_eval_fixture_subprocess(project,split_run,identity)
    resumed,_=v4_eval_run(project,split_run,identity,v4_eval_fixture_players())
    if fresh_digest!=v4_digest(v4_eval_semantic(whole)): raise AssertionError('Fresh process resumed chronology/RNG differs')
    if v4_eval_semantic(whole)!=v4_eval_semantic(resumed): raise AssertionError('Exact resumed chronology/RNG differs')
    retention=v4_read_json(project/'checkpoints'/split_run/'manifest.json')
    if len(retention['entries'])<2 or len(retention['entries'])>3: raise AssertionError('latest2+best retention failed')
    rejected=[]
    bad=copy.deepcopy(identity);bad['protocol']['komi']=6.5
    try: v4_eval_run(project,split_run,bad,v4_eval_fixture_players())
    except RuntimeError: rejected.append('changed_identity')
    else: raise AssertionError('Changed protocol accepted')
    for mode,expected in [('illegal','UNKNOWN_ILLEGAL_MOVE'),('deadline','UNKNOWN_DEADLINE'),('cutoff','UNKNOWN_MOVE_CAP')]:
        test_protocol=V4GNUProtocol(split='recovery_fixture',games=2,seed=910000027,opening_plies=0,max_plies=4)
        test_players=v4_eval_fixture_players(mode);test_identity=v4_eval_identity(project,test_protocol,test_players)
        unknown,_=v4_eval_run(project,'ahr-v4-arena-'+mode+'-'+suffix,test_identity,test_players)
        if any(r['status']!=expected or r['candidate_score'] is not None for r in unknown['results']):
            raise AssertionError('UNKNOWN scoring regression: '+mode)
    # Actual pinned GNU process, full-history replay, independent emitted-move legality,
    # deterministic repeat of the same restored chronology and state-derived seed.
    binary=gnugo10_binary_identity(project)
    opening=initial_state(9,7.5);actions=[40,30]
    for action in actions: opening=play(opening,action)
    implementation=GNU10ReplayPlayer(project,level=1,seed=910000031,command_timeout_s=3.0)
    implementation.prepare_position(opening,[],actions);one=implementation(opening,5.0)
    implementation.prepare_position(opening,[],actions);two=implementation(opening,5.0)
    if one['action']!=two['action'] or one['reference_seed']!=two['reference_seed'] or not one['legality_verified']:
        raise AssertionError('GNU replay recovery changed move/seed')
    if GNUGO10_RULE_FLAGS!=['--chinese-rules','--positional-superko','--forbid-suicide','--capture-all-dead','--never-resign']:
        raise AssertionError('GNU rule flags changed')
    # Check corruption rejection without modifying a valid run.
    corrupt_dir=project/'checkpoints'/('ahr-v4-arena-corrupt-'+suffix)
    with checkpoint_lock(corrupt_dir):
        corrupt=copy.deepcopy(partial);v4_eval_commit(corrupt_dir,corrupt,v4_eval_fixture_players(),first=True)
    manifest=v4_read_json(corrupt_dir/'manifest.json');entry=manifest['entries'][0]
    target=corrupt_dir/entry['file'];data=bytearray(target.read_bytes());data[-1]^=1;atomic_write_bytes(target,data)
    try: v4_eval_strict_load(corrupt_dir)
    except RuntimeError: rejected.append('corrupt_payload')
    else: raise AssertionError('Corrupted checkpoint accepted')
    receipt={'schema':V4_EVAL_SCHEMA,'status':'PASS','exact_resume':True,
        'sources':v4_source_identity(project,require_all=False),'checks':{'midgame_rng_chronology':True,'fresh_process_resume':True,'unknown_semantics':True,
        'latest2_best_retention':True,'immutable_request':True,'corrupt_payload_rejected':True,
        'gnu_pinned_binary':True,'gnu_chronological_repeat':True,'rules_and_color_pairs':True},
        'gnu_identity':binary,'gnu_smoke_elapsed_s':[one['elapsed_s'],two['elapsed_s']],
        'fixture_scope':'recovery wiring only; no playing-strength claim; actual V4 model validated separately in 01/02',
        'fixture_runs':['ahr-v4-arena-whole-'+suffix,split_run],
        'device_for_strength_runs':device,'resume_cell':V4_EVAL_RESUME}
    v4_atomic_json(v4_report_path(project,'arena_validation.json'),receipt)
    return receipt


In [ ]:
validation = v4_validate_arena_resume(project, device='cuda:1')
print(json.dumps({key:validation[key] for key in ('status','exact_resume','checks','gnu_smoke_elapsed_s')}, indent=2))


## Launch and interpret results

Notebook 04 calls `v4_run_gnu_evaluation` for 32-game development blocks and optionally a fresh 64-game confirmation. Select a protected development checkpoint before freezing the final 200-game request. Resume uses the same evaluation ID and resolves its existing protected model rather than whatever training checkpoint is latest.

A completed benchmark can satisfy `observed_half_wins` while its interval still includes 50%. The interval treats the two color-swapped games as one opening block. Unknown and pending games retain possible outcome bounds and cannot satisfy the all-valid success criterion. Group-loss diagnostics are descriptive and do not prove the first bad move. These results establish measured play under this protocol, not full-game optimality.